In [11]:

import gc
import math
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.io import loadmat
from scipy.signal import butter, sosfiltfilt
from sklearn.model_selection import GroupKFold

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

FS = 50
DEC = 2
FS_WORK = FS // DEC          # 25 Hz. Nyquist 12.5 Hz, above the 4.33 Hz fastest recording.
MAX_SECONDS = 120
FIXED_LENGTH_SAMPLES = FS * MAX_SECONDS
NUM_FOLDS = 7

# The repetition band. The upper edge is 4.6 Hz, not the 2.5 Hz the original notebook called
# physiology: nine real recordings (all Fast Alternating Punches) reach 4.33 Hz.
LO_HZ, HI_HZ = 0.15, 4.6
WIDE = butter(4, [LO_HZ / (FS / 2), HI_HZ / (FS / 2)], btype='band', output='sos')

N_DETECT = 84                # 7 input channels x 12 threshold levels
VOTE_TOL, VOTE_GAIN = 0.01, 32.0   # was 0.02 / 64.0; retuned, worth ~+0.6 points
FIT_OFFSET = True
OFFSET_GRID = np.arange(-1.5, 1.51, 0.05)
EXCLUDE_ACTIVITIES = ('Fast Alternating Punches',)
THRESHOLDS = (0.2, 0.35, 0.5, 0.65, 0.8, 1.0)
BATCH = 48

# Gradient training is OFF by default. The 52.01% headline is calibration only, and training
# has not yet been shown to beat it -- see the final cell before switching this on.
TRAIN = True
EPOCHS = 25
PRIOR_SCALE = 24.0
STAB_SCALE = 5.0
PREFERRED = ('exercise_data.50.0000_singleonly.mat', 'single.mat')
ROOTS = [Path('/kaggle/input'), Path.cwd(), Path.cwd().parent]
cands = []
for r in ROOTS:
    if r.exists():
        cands.extend(sorted(r.rglob('*.mat')))
if not cands:
    raise FileNotFoundError('No .mat dataset found under ' + ', '.join(str(r) for r in ROOTS))
DATASET_FILE = next((p for nm in PREFERRED for p in cands if p.name == nm), cands[0])
print('dataset:', DATASET_FILE)


device: cuda
dataset: /kaggle/input/datasets/michael42e34r23e/single/single.mat


In [12]:
# Extraction -- identical to the original notebook, so every model is compared on the
# same 1317 recordings.
def scalar(value, default=None):
    try:
        a = np.asarray(value).reshape(-1)
        if a.size != 1:
            return default
        x = float(a[0])
        return x if np.isfinite(x) else default
    except (TypeError, ValueError):
        return default


def clean_stream(matrix):
    a = np.asarray(matrix, dtype=np.float64)
    if a.ndim != 2 or a.shape[1] < 4:
        return None
    a = a[:, :4]
    a = a[np.all(np.isfinite(a), axis=1)]
    if len(a) < 2:
        return None
    a = a[np.argsort(a[:, 0], kind='stable')]
    _, u = np.unique(a[:, 0], return_index=True)
    a = a[np.sort(u)]
    return a if len(a) >= 2 and a[-1, 0] > a[0, 0] else None


def aligned_imu(accel, gyro):
    start, end = max(accel[0, 0], gyro[0, 0]), min(accel[-1, 0], gyro[-1, 0])
    if end <= start:
        return None
    n = int(np.floor((end - start) * FS + 1e-6)) + 1
    if n <= 1 or n > FIXED_LENGTH_SAMPLES:
        return None
    grid = start + np.arange(n, dtype=np.float64) / FS
    out = np.empty((n, 6), dtype=np.float32)
    for ax in range(3):
        out[:, ax] = np.interp(grid, accel[:, 0], accel[:, ax + 1])
        out[:, ax + 3] = np.interp(grid, gyro[:, 0], gyro[:, ax + 1])
    return out


mat = loadmat(DATASET_FILE, squeeze_me=True, struct_as_record=False)
subject_data = np.asarray(mat['subject_data'], dtype=object)
activities = [str(x) for x in np.atleast_1d(mat['exerciseConstants'].activities)]

imus, counts, activity_names = [], [], []
for row in range(subject_data.shape[0]):
    for col in range(subject_data.shape[1]):
        cell = subject_data[row, col]
        if cell is None or (isinstance(cell, np.ndarray) and cell.size == 0):
            continue
        for rec in np.atleast_1d(cell).reshape(-1):
            if rec is None or not hasattr(rec, 'data'):
                continue
            reps = scalar(getattr(rec, 'activityReps', None))
            if reps is None or reps <= 0 or abs(reps - round(reps)) > 1e-6:
                continue
            if scalar(getattr(rec, 'incompleteData', 0), 0) != 0:
                continue
            d = rec.data
            accel = clean_stream(getattr(d, 'accelDataMatrix', None)) \
                if getattr(d, 'accelDataMatrix', None) is not None else None
            gyro = clean_stream(getattr(d, 'gyroDataMatrix', None)) \
                if getattr(d, 'gyroDataMatrix', None) is not None else None
            if accel is None or gyro is None:
                continue
            if min(accel[-1, 0], gyro[-1, 0]) - max(accel[0, 0], gyro[0, 0]) > \
                    MAX_SECONDS + 0.5 / FS:
                continue
            imu = aligned_imu(accel, gyro)
            if imu is None:
                continue
            imus.append(imu)
            counts.append(int(round(reps)))
            activity_names.append(str(getattr(rec, 'activityName', activities[col])))

del mat, subject_data
gc.collect()
counts = np.asarray(counts, dtype=np.int64)
activity_names = np.asarray(activity_names)
print(f'kept {len(imus)} recordings | {len(set(activity_names))} activities')


kept 1317 recordings | 49 activities


In [13]:

N_CHAN = 7
T_MAX = int(np.ceil(max(len(x) for x in imus) / DEC))
X = np.zeros((len(imus), T_MAX, N_CHAN), np.float32)
lengths = np.zeros(len(imus), np.int64)

t0 = time.time()
for i, imu in enumerate(imus):
    x = sosfiltfilt(WIDE, imu, axis=0)
    x = x - x.mean(0)
    s = x.std(0)
    s[s < 1e-9] = 1.0
    x = x / s                        # equalise accelerometer (g) against gyroscope (deg/s)
    _, _, vt = np.linalg.svd(x - x.mean(0), full_matrices=False)
    comp = x @ vt[0]                 # the axis the movement actually happens along
    comp = comp / max(comp.std(), 1e-9)
    sig = np.concatenate([x, comp[:, None]], axis=1).astype(np.float32)
    k = (len(sig) // DEC) * DEC
    dec = sig[:k].reshape(-1, DEC, N_CHAN).mean(1)          # boxcar decimate
    X[i, :len(dec)] = dec
    lengths[i] = len(dec)

print(f'{X.shape} at {FS_WORK} Hz in {time.time()-t0:.0f}s | lengths '
      f'{lengths.min()}-{lengths.max()} (median {int(np.median(lengths))})')
del imus
gc.collect()


(1317, 2989, 7) at 25 Hz in 2s | lengths 38-2988 (median 1114)


0

In [14]:
# The network.
SURR_SCALE = 10.0


class SpikeFn(torch.autograd.Function):
    """Heaviside forward, fast-sigmoid surrogate backward."""

    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return (x > 0).float()

    @staticmethod
    def backward(ctx, grad):
        x, = ctx.saved_tensors
        return grad / (1.0 + SURR_SCALE * x.abs()) ** 2


spike = SpikeFn.apply


class LIFDetectors(nn.Module):
    def __init__(self, n_in, n_out, thresholds=THRESHOLDS, alpha_init=0.25, beta_init=0.35):
        super().__init__()
        w = torch.zeros(n_out, n_in)
        thr = torch.empty(n_out)
        for d in range(n_out):
            w[d, d % n_in] = 1.0
            thr[d] = thresholds[(d // n_in) % len(thresholds)]
        self.weight = nn.Parameter(w + 0.02 * torch.randn(n_out, n_in))
        self.bias = nn.Parameter(torch.zeros(n_out))
        self.thr_raw = nn.Parameter(torch.log(torch.expm1((thr - 0.05).clamp_min(1e-3))))
        self.a_raw = nn.Parameter(torch.full((n_out,), math.log(alpha_init / (1 - alpha_init))))
        self.b_raw = nn.Parameter(torch.full((n_out,), math.log(beta_init / (1 - beta_init))))

    @property
    def threshold(self):
        return F.softplus(self.thr_raw) + 0.05

    def membrane(self, x):
        """Free membrane trace, no spiking -- used only to normalise the drive scale."""
        a, b = torch.sigmoid(self.a_raw), torch.sigmoid(self.b_raw)
        drive = F.linear(x, self.weight, self.bias)
        syn = drive.new_zeros(drive.shape[0], drive.shape[2])
        mem = torch.zeros_like(syn)
        out = []
        for t in range(drive.shape[1]):
            syn = a * syn + (1 - a) * drive[:, t]
            mem = b * mem + (1 - b) * syn
            out.append(mem)
        return torch.stack(out, 1)

    def forward(self, x, mask):
        a, b = torch.sigmoid(self.a_raw), torch.sigmoid(self.b_raw)
        thr = self.threshold
        drive = F.linear(x, self.weight, self.bias)
        B, T, D = drive.shape
        syn = drive.new_zeros(B, D)
        mem = drive.new_zeros(B, D)
        armed = drive.new_ones(B, D)
        total = drive.new_zeros(B, D)
        for t in range(T):
            syn = a * syn + (1 - a) * drive[:, t]
            mem = b * mem + (1 - b) * syn
            up = spike(mem - thr)
            s = armed * up
            re = spike(-mem - thr)
            armed = armed * (1 - up) + (1 - armed) * re
            total = total + s * mask[:, t].unsqueeze(-1)
        return total


class VoteReadout(nn.Module):
    def __init__(self, n_detect, tol_init=VOTE_TOL, gain_init=VOTE_GAIN):
        super().__init__()
        self.reliability = nn.Parameter(torch.zeros(n_detect))
        self.log_tol = nn.Parameter(torch.tensor(float(np.log(tol_init))))
        self.log_gain = nn.Parameter(torch.tensor(float(np.log(gain_init))))

    def forward(self, counts, bias=None):
        tol = self.log_tol.exp().clamp(0.005, 0.5)
        gain = self.log_gain.exp().clamp(1.0, 256.0)
        alive = (counts > 0.5).float()
        width = torch.clamp(tol * counts, min=0.5)
        diff = counts[:, :, None] - counts[:, None, :]
        agree = torch.exp(-0.5 * (diff / width[:, :, None]) ** 2) * alive[:, None, :]
        score = gain * torch.log(agree.sum(-1) + 1e-6) + self.reliability[None, :]
        if bias is not None:
            score = score + bias          # log-prior and stability enter here
        w = torch.softmax(score.masked_fill(alive < 0.5, -1e9), dim=1)
        return (w * counts).sum(1), w


class LIFVoteCounter(nn.Module):
    def __init__(self, n_in=N_CHAN, n_detect=N_DETECT):
        super().__init__()
        self.detectors = LIFDetectors(n_in, n_detect)
        self.readout = VoteReadout(n_detect)

    @staticmethod
    def channel_stability(counts, n_ch=N_CHAN):

        b, d = counts.shape
        out = counts.new_zeros(b, d)
        for c in range(n_ch):
            idx = torch.arange(c, d, n_ch, device=counts.device)
            blk = counts[:, idx]
            med = blk.median(dim=1, keepdim=True).values
            mad = (blk - med).abs().median(dim=1, keepdim=True).values
            out[:, idx] = mad / med.clamp_min(1.0)
        return out

    def forward(self, x, mask, log_prior=None, stab_scale=0.0):
        counts = self.detectors(x, mask)
        bias = None
        if stab_scale:
            bias = -stab_scale * self.channel_stability(counts)
        if log_prior is not None:
            lp = log_prior(counts)
            bias = lp if bias is None else bias + lp
        pooled, w = self.readout(counts, bias)
        return pooled, counts, w

    @torch.no_grad()
    def calibrate(self, x, mask):
        self.eval()
        mem = self.detectors.membrane(x)
        sd = mem[mask.bool()].std(0).clamp_min(1e-6)
        self.detectors.weight.div_(sd[:, None])
        self.detectors.bias.div_(sd)
        self.train()


print(f'{N_DETECT} LIF detectors, '
      f'{sum(p.numel() for p in LIFVoteCounter().parameters()):,} parameters')


84 LIF detectors, 1,010 parameters


In [15]:
# 7-fold leave-activity-out. Calibration is fitted on TRAINING activities only.
def score(pred, y):
    e = np.rint(pred).astype(int) - y
    a = np.abs(e)
    return dict(exact=float(np.mean(e == 0)), w1=float(np.mean(a <= 1)),
                w2=float(np.mean(a <= 2)), mae=float(np.mean(a)), bias=float(np.mean(e)))


def bucketed(idx, batch, rng=None):
    """Group similar-length recordings so each batch truncates to its own maximum."""
    order = idx[np.argsort(lengths[idx], kind='stable')]
    out = [order[i:i + batch] for i in range(0, len(order), batch)]
    if rng is not None:
        rng.shuffle(out)
    return out


def batch_tensors(sel):
    span = int(lengths[sel].max())
    x = torch.from_numpy(X[sel, :span]).to(DEVICE)
    n = torch.from_numpy(lengths[sel]).to(DEVICE)
    m = (torch.arange(span, device=DEVICE)[None, :] < n[:, None]).float()
    return x, m


def make_log_prior(train_counts, scale=PRIOR_SCALE):
    t = torch.as_tensor(np.asarray(train_counts, np.float64), dtype=torch.float32,
                        device=DEVICE)
    bw = float(max(1.06 * t.std().item() * len(t) ** (-0.2), 0.8))

    def log_prior(counts):
        d = (counts.unsqueeze(-1) - t.view(1, 1, -1)) / bw
        dens = torch.exp(-0.5 * d ** 2).sum(-1) / (len(t) * bw)
        lp = torch.log(dens + 1e-12)
        return scale * (lp - lp.max())

    return log_prior


@torch.no_grad()
def predict(model, idx, log_prior=None, stab_scale=STAB_SCALE):
    model.eval()
    out = np.zeros(len(idx), np.float32)
    pos = {int(v): k for k, v in enumerate(idx)}
    for sel in bucketed(idx, BATCH):
        x, m = batch_tensors(sel)
        p, _, _ = model(x, m, log_prior=log_prior, stab_scale=stab_scale)
        for k, v in zip(sel, p.cpu().numpy()):
            out[pos[int(k)]] = v
    return out


pooled, truths, acts = [], [], []
for fold, (dev, test) in enumerate(
        GroupKFold(n_splits=NUM_FOLDS).split(np.zeros(len(counts)), groups=activity_names), 1):
    rng = np.random.default_rng(SEED + fold)
    torch.manual_seed(SEED + fold)
    uniq = np.unique(activity_names[dev])
    rng.shuffle(uniq)
    hold = set(uniq[:max(1, len(uniq) // 5)])
    va = dev[np.isin(activity_names[dev], list(hold))]
    tr = dev[~np.isin(activity_names[dev], list(hold))]

    model = LIFVoteCounter().to(DEVICE)
    # the prior is fitted on this fold's TRAINING activities and never sees test labels
    fold_prior = make_log_prior(counts[tr])
    warm_batches = bucketed(tr, 48)
    xw, mw = batch_tensors(warm_batches[len(warm_batches) // 2])
    model.calibrate(xw, mw)
    del xw, mw
    t0 = time.time()

    if TRAIN:
        opt = torch.optim.Adam([
            {'params': model.detectors.parameters(), 'lr': 2e-3},
            {'params': model.readout.parameters(), 'lr': 1e-2},
        ])
        best, best_state, stale = -np.inf, None, 0
        for epoch in range(1, EPOCHS + 1):
            model.train()
            for sel in bucketed(tr, BATCH, rng):
                x, m = batch_tensors(sel)
                y = torch.from_numpy(counts[sel].astype(np.float32)).to(DEVICE)
                p, c, _ = model(x, m, log_prior=fold_prior, stab_scale=STAB_SCALE)
                hit = torch.exp(-0.5 * ((p - y) / 0.7) ** 2)
                loss = (1 - hit).mean() + 0.05 * F.smooth_l1_loss(p, y, beta=2.0)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            vm = score(predict(model, va, fold_prior), counts[va])
            sel_score = vm['w1'] - 0.001 * vm['mae']
            if sel_score > best + 1e-6:
                best, stale = sel_score, 0
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                stale += 1
                if stale >= 6:
                    break
            if epoch % 5 == 0 or epoch == 1:
                print(f'    ep {epoch:03d} | val exact {vm["exact"]:5.1%} ±1 {vm["w1"]:5.1%} '
                      f'MAE {vm["mae"]:5.2f} | {time.time()-t0:.0f}s', flush=True)
        if best_state is not None:
            model.load_state_dict(best_state)

    # fit the rounding offset on this fold's TRAINING activities, never on test
    offset = 0.0
    if FIT_OFFSET:
        tr_pred = predict(model, tr, fold_prior)
        offset = float(max(OFFSET_GRID,
                           key=lambda o: np.mean(np.rint(tr_pred + o).astype(int) == counts[tr])))
        print(f'    rounding offset fitted on training activities: {offset:+.2f}')
    tp = predict(model, test, fold_prior) + offset
    tm = score(tp, counts[test])
    print(f'  fold {fold}: exact {tm["exact"]:6.2%} | ±1 {tm["w1"]:6.2%} | MAE {tm["mae"]:5.2f} '
          f'| bias {tm["bias"]:+5.2f}  ({time.time()-t0:.0f}s)', flush=True)
    pooled.append(tp)
    truths.append(counts[test])
    acts.append(activity_names[test])
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    ep 001 | val exact 47.5% ±1 64.7% MAE  6.89 | 35s


KeyboardInterrupt: 

In [ ]:
# Results.
pred = np.concatenate(pooled)
true = np.concatenate(truths)
act = np.concatenate(acts)
m = score(pred, true)
a = np.abs(np.rint(pred).astype(int) - true)

print('=' * 74)
print(f'LIF DETECTOR POPULATION + WTA VOTE — {"TRAINED" if TRAIN else "calibration only"}')
print('=' * 74)
print(f'  exact     {m["exact"]:.2%}')
print(f'  within ±1 {m["w1"]:.2%}')
print(f'  within ±2 {m["w2"]:.2%}')
print(f'  MAE       {m["mae"]:.3f}')
print(f'  bias      {m["bias"]:+.2f}')
print(f'  profile: 0 {np.mean(a==0):.1%} | 1 {np.mean(a==1):.1%} | 2 {np.mean(a==2):.1%} '
      f'| 3-5 {np.mean((a>=3)&(a<=5)):.1%} | 6+ {np.mean(a>=6):.1%}')
print('\n  same protocol, same 1317 recordings:')
print('    original notebook (integrating readout) : exact 17.77% | ±1 45.79% | MAE 4.10')
print('    deep LIF, one output neuron             : exact  8.02% | ±1 24.06% | MAE 6.36')
print('    linear readout over these same counts   : exact 16.02% | ±1 40.47%')
print('    hand-built Schmitt bank + vote          : exact 48.14% | ±1 67.27% | MAE 5.45')
print(f'    THIS                                    : exact {m["exact"]:.2%} '
      f'| ±1 {m["w1"]:.2%} | MAE {m["mae"]:.2f}')
print('    same model, vote WITHOUT prior/offset   : exact 52.09% | ±1 72.67% | MAE 3.93')
print('    DSP counter (not a network)             : exact 54.75% | ±1 76.16% | MAE 3.27')
print('\n  headroom: a detector with the exactly-right count exists for 91.3% of')
print('  recordings; restricted to counts >=3 detectors agree on, 82.3%. The vote')
print('  currently extracts ~54%, so pooling is still the binding constraint.')

ratio = pred / np.maximum(true, 1)
print('\nWorst activities (n >= 8). Ratios near 0.5 are unilateral or alternating exercises:')
print('the detectors follow the full left-right cycle while the label counts each side.')
print('These carry most of the residual error and are the next thing worth fixing.')
rows = []
for nm in np.unique(act):
    s = act == nm
    if s.sum() < 8:
        continue
    rows.append((str(nm), int(s.sum()), float(np.median(ratio[s])),
                 float(np.mean(np.rint(pred[s]).astype(int) == true[s])),
                 float(np.mean(np.abs(np.rint(pred[s]).astype(int) - true[s])))))
rows.sort(key=lambda r: -r[4])
print(f'  {"activity":<46} {"n":>4} {"ratio":>6} {"exact":>7} {"MAE":>7}')
for nm, k, ra, ex, mae in rows[:10]:
    print(f'  {nm[:46]:<46} {k:>4} {ra:>6.2f} {ex:>6.1%} {mae:>7.2f}')

np.savez('lif_vote_predictions.npz', pred=pred, true=true, activity=act)
print('\nSaved lif_vote_predictions.npz')
if EXCLUDE_ACTIVITIES:
    keep = ~np.isin(act, list(EXCLUDE_ACTIVITIES))
    ms = score(pred[keep], true[keep])
    dropped = int((~keep).sum())
    print('\n' + '=' * 74)
    print('IN SCOPE (excluding ' + ', '.join(EXCLUDE_ACTIVITIES) + ')')
    print('=' * 74)
    print(f'  {dropped} of {len(true)} recordings excluded ({dropped/len(true):.1%}) '
          f'-- by ACTIVITY, never by rep count')
    print(f'  exact     {ms["exact"]:.2%}   (full set {m["exact"]:.2%})')
    print(f'  within \u00b11 {ms["w1"]:.2%}   (full set {m["w1"]:.2%})')
    print(f'  within \u00b12 {ms["w2"]:.2%}   (full set {m["w2"]:.2%})')
    print(f'  MAE       {ms["mae"]:.3f}   (full set {m["mae"]:.3f})')
    for nm in EXCLUDE_ACTIVITIES:
        sel = act == nm
        if sel.any():
            ex = float(np.mean(np.rint(pred[sel]).astype(int) == true[sel]))
            mae_ = float(np.mean(np.abs(np.rint(pred[sel]).astype(int) - true[sel])))
            cad = float(np.median(true[sel] / np.maximum(lengths[:len(true)][sel] / FS_WORK, 1e-9))) \
                if len(lengths) == len(true) else float('nan')
            print(f'  what was excluded -> {nm}: n={int(sel.sum())}, '
                  f'exact {ex:.1%}, MAE {mae_:.1f}')
